# Notebook 07 — Latent Structure

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 01 treated **gain** as the first measurable context signal.

Notebook 07 asks the next question:

> If gain appeared, what reusable structure was the system learning?

CL-BENCH is built around tasks where sequential instances share hidden structure: database schemas, codebase layouts, transmitter maps, disease dynamics, opponent policies, and demand patterns.

This notebook builds a lightweight latent-structure map and connects it to gain behavior.

Outputs:

- `results/07_latent_structure_map.csv`
- `results/07_structure_transfer_matrix.csv`
- `results/07_latent_structure_summary.json`
- `figures/07_task_structure_map.png`
- `figures/07_structure_transfer_matrix.png`
- `figures/07_context_transition_costs.png`
- `results/07_latent_structure_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain

print("Imports complete.")

## 2. Load Prior Gain Artifacts

Notebook 07 uses the trajectory produced by Notebook 01 when available:

```text
results/01_gain_signal_trajectory.csv
```

If it is missing, the notebook falls back to the Notebook 00 trajectory or a small synthetic trajectory.

In [ ]:
trajectory_01 = RESULTS_DIR / "01_gain_signal_trajectory.csv"
trajectory_00 = RESULTS_DIR / "00_context_gain.csv"

if trajectory_01.exists():
    gain_df = pd.read_csv(trajectory_01)
    source_file = trajectory_01
    print("Loaded:", trajectory_01)
elif trajectory_00.exists():
    gain_df = pd.read_csv(trajectory_00)
    source_file = trajectory_00
    print("Loaded:", trajectory_00)
else:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    gain_df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    gain_df["gain"] = compute_gain(
        gain_df["stateful_reward"].tolist(),
        gain_df["stateless_reward"].tolist(),
    )
    source_file = None

if "gain" not in gain_df.columns:
    gain_df["gain"] = compute_gain(
        gain_df["stateful_reward"].tolist(),
        gain_df["stateless_reward"].tolist(),
    )

if "cumulative_gain" not in gain_df.columns:
    gain_df["cumulative_gain"] = gain_df["gain"].cumsum()

if "gain_delta" not in gain_df.columns:
    gain_df["gain_delta"] = gain_df["gain"].diff().fillna(0.0)

gain_df

## 3. Define Latent Structure

A **latent structure** is a reusable regularity that is not explicitly given to the agent, but can be inferred through sequential experience.

Examples:

- hidden schema conventions
- codebase topology
- persistent transmitters
- opponent strategies
- cross-location demand dynamics
- disease survival factors

In CL-BENCH, gain is expected to arise when a system discovers and reuses these structures.

In [ ]:
structure_df = pd.DataFrame([
    {
        "task": "Database Exploration",
        "domain": "database analytics",
        "latent_structure": "schema layout, table naming conventions, column encodings",
        "structure_type": "symbolic",
        "reuse_action": "reuse discovered schema facts to answer later questions with fewer queries",
        "failure_mode": "stale schema reuse after migration",
    },
    {
        "task": "Codebase Adaptation",
        "domain": "software engineering",
        "latent_structure": "file layout, module boundaries, test commands, contributor conventions",
        "structure_type": "symbolic",
        "reuse_action": "reuse repo navigation and test patterns across related issues",
        "failure_mode": "overfitting to previous file locations or test patterns",
    },
    {
        "task": "Blind Spectrum Monitoring",
        "domain": "signal processing",
        "latent_structure": "persistent transmitter center frequencies and bandwidths",
        "structure_type": "physical",
        "reuse_action": "carry forward dormant transmitters not visible in the current scan",
        "failure_mode": "forgetting dormant channels or hallucinating inactive ones",
    },
    {
        "task": "Cohort Studies",
        "domain": "epidemiological forecasting",
        "latent_structure": "patient mixture, risk factors, biased cohorts, survival dynamics",
        "structure_type": "statistical",
        "reuse_action": "integrate cross-study evidence into population survival estimates",
        "failure_mode": "spurious correlations mistaken for stable predictors",
    },
    {
        "task": "Exploitable Poker",
        "domain": "strategic game-playing",
        "latent_structure": "opponent archetypes, betting tendencies, stage-specific policies",
        "structure_type": "behavioral",
        "reuse_action": "infer fixed opponent policy and exploit repeated betting patterns",
        "failure_mode": "overfitting to recent hands or failing to transfer opponent model",
    },
    {
        "task": "Sales Prediction",
        "domain": "demand forecasting",
        "latent_structure": "cross-location product-cluster growth rates",
        "structure_type": "statistical",
        "reuse_action": "transfer demand patterns across stores and products",
        "failure_mode": "misreading local noise as global trend",
    },
])

structure_df

## 4. Structure Type Counts

This figure summarizes what kinds of reusable structures CL-BENCH tasks require.

In [ ]:
type_counts = (
    structure_df["structure_type"]
    .value_counts()
    .rename_axis("structure_type")
    .reset_index(name="count")
)

fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(type_counts["structure_type"], type_counts["count"])
ax.set_title("Notebook 07: CL-BENCH Latent Structure Types")
ax.set_xlabel("Structure Type")
ax.set_ylabel("Task Count")
ax.grid(True, axis="y", alpha=0.3)

task_structure_path = FIGURES_DIR / "07_task_structure_map.png"
fig.tight_layout()
fig.savefig(task_structure_path, dpi=160)

type_counts

## 5. Structure Transfer Matrix

The transfer matrix maps tasks to broad structure types.

This is not a claim that the tasks are reducible to one type. It is a first-pass map of the dominant reusable structure each task requires.

In [ ]:
structure_types = ["symbolic", "behavioral", "statistical", "physical"]

transfer_matrix = pd.DataFrame(
    0.0,
    index=structure_df["task"],
    columns=structure_types,
)

for _, row in structure_df.iterrows():
    transfer_matrix.loc[row["task"], row["structure_type"]] = 1.0

transfer_matrix

## 6. Figure — Structure Transfer Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

matrix_values = transfer_matrix.values
im = ax.imshow(matrix_values, aspect="auto")

ax.set_xticks(np.arange(len(transfer_matrix.columns)))
ax.set_yticks(np.arange(len(transfer_matrix.index)))

ax.set_xticklabels(transfer_matrix.columns, rotation=30, ha="right")
ax.set_yticklabels(transfer_matrix.index)

for i in range(matrix_values.shape[0]):
    for j in range(matrix_values.shape[1]):
        ax.text(j, i, f"{matrix_values[i, j]:.0f}", ha="center", va="center")

ax.set_title("Notebook 07: Structure Transfer Matrix")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

transfer_matrix_path = FIGURES_DIR / "07_structure_transfer_matrix.png"
fig.tight_layout()
fig.savefig(transfer_matrix_path, dpi=160)

transfer_matrix_path

## 7. Context Transition Costs

Notebook 01 showed gain accumulation.

Notebook 07 checks where gain drops, because drops often occur when:

- the context changes,
- the learned structure no longer transfers,
- stale assumptions need revision.

The largest negative `gain_delta` is a first-pass transition-cost signal.

In [ ]:
transition_df = gain_df.copy()

transition_df["transition_cost"] = transition_df["gain_delta"].apply(
    lambda x: abs(x) if x < 0 else 0.0
)

largest_drop_idx = transition_df["gain_delta"].idxmin()
largest_drop = transition_df.loc[largest_drop_idx].to_dict()

transition_df[[
    "instance",
    "variant",
    "gain",
    "cumulative_gain",
    "gain_delta",
    "transition_cost",
]]

## 8. Figure — Context Transition Costs

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    transition_df["instance"],
    transition_df["transition_cost"],
)

for boundary in transition_df.groupby("variant")["instance"].min().tolist()[1:]:
    ax.axvline(boundary - 0.5, linestyle=":", linewidth=1)

ax.set_title("Notebook 07: Context Transition Costs")
ax.set_xlabel("Instance")
ax.set_ylabel("Negative Gain Delta Magnitude")
ax.grid(True, axis="y", alpha=0.3)

transition_cost_path = FIGURES_DIR / "07_context_transition_costs.png"
fig.tight_layout()
fig.savefig(transition_cost_path, dpi=160)

transition_cost_path

## 9. Summary

Notebook 07 connects gain to latent structure.

The key distinction:

- Notebook 01 asks whether context helped.
- Notebook 07 asks what kind of structure made context useful.

In [ ]:
structure_summary = {
    "total_tasks_mapped": int(len(structure_df)),
    "structure_type_counts": {
        row["structure_type"]: int(row["count"])
        for _, row in type_counts.iterrows()
    },
    "dominant_structure_types": type_counts["structure_type"].tolist(),
    "trajectory_source": str(source_file) if source_file is not None else "fallback",
    "largest_gain_drop_instance": int(largest_drop["instance"]),
    "largest_gain_drop_variant": str(largest_drop["variant"]),
    "largest_gain_drop": float(largest_drop["gain_delta"]),
    "largest_transition_cost": float(largest_drop["transition_cost"]),
    "final_cumulative_gain": float(transition_df["cumulative_gain"].iloc[-1]),
}

structure_summary

## 10. Export Artifacts

In [ ]:
structure_map_path = RESULTS_DIR / "07_latent_structure_map.csv"
transfer_matrix_path_csv = RESULTS_DIR / "07_structure_transfer_matrix.csv"
transition_costs_path = RESULTS_DIR / "07_context_transition_costs.csv"
summary_path = RESULTS_DIR / "07_latent_structure_summary.json"
zip_path = RESULTS_DIR / "07_latent_structure_artifacts.zip"

structure_df.to_csv(structure_map_path, index=False)
transfer_matrix.to_csv(transfer_matrix_path_csv)
transition_df.to_csv(transition_costs_path, index=False)

summary_with_metadata = {
    **structure_summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "07_latent_structure.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        structure_map_path,
        transfer_matrix_path_csv,
        transition_costs_path,
        summary_path,
        task_structure_path,
        transfer_matrix_path,
        transition_cost_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    structure_map_path,
    transfer_matrix_path_csv,
    transition_costs_path,
    summary_path,
    task_structure_path,
    transfer_matrix_path,
    transition_cost_path,
    zip_path,
]:
    print("-", path)

## 11. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/07_latent_structure_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 12. Notebook 07 Claim

Gain measures that learning happened.

Latent structure explains why learning happened.

\[
\text{experience}
\rightarrow
\text{structure discovery}
\rightarrow
\text{context reuse}
\rightarrow
\text{gain}
\]

In RML terms:

> Continual learning is not merely remembering prior observations. It is discovering reusable structure within changing contexts.

Next notebook:

```text
11_state_vs_stateless.ipynb
```

will return to the central CL-BENCH comparison and analyze where accumulated state helps or hurts.